# Why `row_extra_quote0_col2.csv` loads 0 rows

The injected quote makes the **header** strict-invalid. With `strict_mode=true`, DuckDB silently returns 0 rows (0 rejects, no error) even though the header is skipped.

In [1]:
import duckdb

F = "/home/robin/offsite/pollock-offsite/data/small_sample/csv/row_extra_quote0_col2.csv"
COLS = "{" + ", ".join(f"'_c{i}': 'VARCHAR'" for i in range(9)) + "}"

def load(path=F, escape='"', strict=True):
    q = (f"SELECT count(*) FROM read_csv(?, auto_detect=false, columns={COLS}, "
         "delim=',', quote='\"', escape=?, header=false, skip=1, "
         "strict_mode=?, null_padding=true)")
    return duckdb.execute(q, [path, escape, strict]).fetchone()[0]

for esc in ('"', ''):
    for strict in (True, False):
        print(f"escape={esc!r:4} strict={strict!s:5} -> {load(escape=esc, strict=strict)} rows")

escape='"'  strict=True  -> 0 rows
escape='"'  strict=False -> 81 rows
escape=''   strict=True  -> 0 rows
escape=''   strict=False -> 83 rows


In [2]:
# Fix ONLY the header's extra quote -> all rows load, even under strict mode
fixed = "/tmp/fixed_header.csv"
body = open(F, "rb").read().split(b"\r\n", 1)[1]
open(fixed, "wb").write(b"DATE,TIME,Qty,PRODUCTID,Price,ProductType,ProductDescription,URL,Comments\r\n" + body)

print("fixed header, strict ->", load(path=fixed), "rows (expected 83)")

fixed header, strict -> 83 rows (expected 83)
